<H3>PRI 2024/25: first project delivery</H3>

**GROUP 9**
- Diogo Dinis, 99066
- Afonso Lança, 103838
- Luís Calado, 103883

<H3>Part I: demo of facilities</H3>

In [ ]:
!pip install pytrec_eval
!pip install ir_datasets

In [ ]:
# --- Import Libraries ---

import ir_datasets
import pytrec_eval
from time import time
import sys
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from collections import Counter, defaultdict, namedtuple
import math
import re
import matplotlib.pyplot as plt

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

In [ ]:
# --- Load Dataset ---
dataset = ir_datasets.load("cord19/trec-covid")

queries = [query for query in dataset.queries_iter()]

qrels = {}

for qrel in dataset.qrels_iter():
    qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

docs = [doc for doc in dataset.docs_iter()]

A) **Indexing** (preprocessing and indexing options)

In [ ]:
# --- Preprocessing and Indexing Functions --

stopwords_set = set(stopwords.words("english"))
non_alpha_numeric = re.compile(r'[^a-z0-9\s]')
def preprocess(text, stopwords=stopwords_set):
    """Normalize and preprocess a text string."""
    text = text.lower()
    text = non_alpha_numeric.sub(" ", text)
    words = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stopwords]
    return words  # Return list of tokens

def calculate_index_size(postings):
    """Calculate an approximate memory size of the postings dictionary."""
    space_required = sys.getsizeof(postings)  # Base dictionary size
    for term, doc_dict in postings.items():
        space_required += sys.getsizeof(term)    # Term size
        space_required += sys.getsizeof(doc_dict)  # Dict size for postings
        for docid, tf in doc_dict.items():
            space_required += sys.getsizeof(docid)
            space_required += sys.getsizeof(tf)
    return space_required

def preprocess_corpus(docs, stop_words, preprocess_text):
    preprocessed_docs = {}
    for doc in docs:
        docid = doc.doc_id
        # Combine title and abstract.
        doc_text = f"{doc.title} {doc.abstract}"
        if preprocess_text:
            tokens = preprocess(doc_text, stop_words)
        else:
            tokens = doc_text.split()
        preprocessed_docs[docid] = tokens
    return preprocessed_docs

def build_inverted_index_from_tokens(preprocessed_docs):
    postings = defaultdict(dict)
    doc_len = defaultdict(int)
    for docid, tokens in preprocessed_docs.items():
        doc_vector = Counter(tokens)
        for term, tf in doc_vector.items():
            postings[term][docid] = tf
            doc_len[docid] += tf
    return postings, doc_len

def indexing(docs, stopwords_lang="english", preprocess_text=True):
    stop_words = set(stopwords.words(stopwords_lang))
    TfIndex = namedtuple("tf_index", "postings doc_len avg_doc_len")

    start_preprocess = time()
    preprocessed_docs = preprocess_corpus(docs, stop_words, preprocess_text)
    preprocessing_time = time() - start_preprocess

    start_indexing = time()
    postings, doc_len = build_inverted_index_from_tokens(preprocessed_docs)
    indexing_time = time() - start_indexing

    # Compute overall index statistics.
    N = len(preprocessed_docs)
    total_length = sum(doc_len.values())
    avg_doc_len = total_length / N if N > 0 else 0
    indexing_space = calculate_index_size(postings)

    return TfIndex(postings, doc_len, avg_doc_len), preprocessing_time, indexing_time, indexing_space

In [ ]:
# --- Preprocessing Demo ---
sample_text = "COVID-19, also known as the coronavirus, has impacted millions worldwide! The pandemic's effects are unprecedented."

tokens = preprocess(sample_text)

print("Original Text:")
print(sample_text)
print("\nPreprocessed Tokens:")
print(tokens)

In [ ]:
# --- Indexing Demo ---
index, preprocessing_time, indexing_time, space_used = indexing(docs)

print(f"Preprocessing time: {preprocessing_time:.4f} seconds")
print(f"Indexing time: {indexing_time:.4f} seconds")
print(f"Approximate index space used: {space_used / (1024 * 1024):.2f} MB")

B) **Boolean Query**

In [ ]:
# --- Boolean Retrieval Functions ---

def boolean_query(q, I, Q, use_description=False, use_narrative=False):
    # Retrieve the query by its id.
    query = next((query for query in Q if query.query_id == q), None)
    if query is None:
        return set()

    query_text = query.title
    if use_description:
        query_text += f" {query.description}"
    if use_narrative:
        query_text += f" {query.narrative}"

    stop_words = set(stopwords.words("english"))
    processed_query = preprocess(query_text, stop_words)

    # query_terms = processed_query.split()
    if not processed_query:
        return set()

    hits = None
    for term in processed_query:
        if term in I.postings:
            term_docs = set(I.postings[term].keys())
            hits = term_docs if hits is None else hits & term_docs  # AND logic.
        else:
            return set()  # If any term is missing, return empty set.

    return sorted(hits) if hits else set()

In [ ]:
# --- Boolean Retrieval Demo ---
sample_query = queries[0]
print("Sample Query ID:", sample_query.query_id)
print("Query Title:", sample_query.title)
print("Query Description:", sample_query.description)
print("Query Narrative:", sample_query.narrative)

retrieved_docs = boolean_query(sample_query.query_id, index, queries, use_description=False, use_narrative=False)

print("\nNumber of retrieved documents:", len(retrieved_docs))
print("Document IDs for the first five retrieved documents:", retrieved_docs[:5])

if retrieved_docs:
    first_doc_id = retrieved_docs[0]
    first_doc = next((doc for doc in docs if doc.doc_id == first_doc_id), None)
    if first_doc:
        print("\nFirst Retrieved Document Details:")
        print("\tTitle:", first_doc.title)
        print("\tAbstract:", first_doc.abstract)
    else:
        print("The first document was not found in the docs collection.")
else:
    print("No documents retrieved for the query.")

C) **Ranking**

In [ ]:
# --- Ranking Functions ---

def tf(t, d, I, doc_len, weighting_tf="n", **kwargs):
    f_td = I.postings[t].get(d, 0)
    if f_td == 0:
        return 0
    if weighting_tf == "n":
        return f_td / doc_len
    elif weighting_tf == "l":
        return 1 + math.log(f_td)

def idf(t, I, N, weighting_idf="t", **kwargs):
    df = len(I.postings[t])
    if weighting_idf == "n":
        return 1
    elif weighting_idf == "t":
        return math.log(N / df)
    elif weighting_idf == "p":
        return max(0, math.log((N - df) / df))

def ranking(q, p, I, scoring_function, **kwargs):
    stop_words = set(stopwords.words("english"))
    query_terms = preprocess(q, stop_words)

    # Only consider docs that contain at least one query term.
    candidate_docs = set()
    for qt in query_terms:
        if qt in I.postings:
            candidate_docs.update(I.postings[qt].keys())

    scores = []
    N = len(I.doc_len)
    for d in candidate_docs:
        score = 0
        for qt in query_terms:
            if qt in I.postings and d in I.postings[qt]:
                score += scoring_function(qt, d, I, N, I.doc_len[d], avg_doc_len=I.avg_doc_len, **kwargs)
        scores.append((d, score))
    ranked_results = sorted(scores, key=lambda x: x[1], reverse=True)[:p]
    return ranked_results

*C.1 IR models (TF-IDF, BM25)*

In [ ]:
# --- Ranked IR Models ---

def tf_idf(t, d, I, N, doc_len, weighting_tf="l", weighting_idf="t", **kwargs):
    return tf(t, d, I, doc_len, weighting_tf=weighting_tf) * idf(t, I, N, weighting_idf=weighting_idf)

def bm25(t, d, I, N, doc_len, k1=1.1, b=0.4, **kwargs):
    avg_doc_len = kwargs.get('avg_doc_len', 1)
    idf_value = idf(t, I, N)
    f_td = I.postings[t].get(d, 0)
    if f_td == 0:
        return 0
    numerator = idf_value * f_td * (k1 + 1)
    denominator = f_td + k1 * (1 - b + b * (doc_len / avg_doc_len))
    return numerator / denominator

def reciprocal_rank_fusion(query, num_retrieval, I, scoring_functions, y=50, **kwargs):
    scores = {}
    for scoring_func_name, scoring_func in scoring_functions.items():
        ranked_results = ranking(query.title, num_retrieval, I, scoring_func, **kwargs)
        for i, (docid, _) in enumerate(ranked_results):
            if docid not in scores:
                scores[docid] = 0
            scores[docid] += 1 / (i + y)

    ranked_rrf = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:num_retrieval]
    return ranked_rrf

In [ ]:
# --- Ranking Demo ---
first_query = queries[0]
top_n = 5

print("Query ID:", first_query.query_id)
print("Query Title:", first_query.title)

print("\nTF-IDF Ranking:")
ranked_tf_idf = ranking(first_query.title, top_n, index, tf_idf)
for doc_id, score in ranked_tf_idf:
    print("Doc ID:", doc_id, "Score:", score)

print("\nBM25 Ranking:")
ranked_bm25 = ranking(first_query.title, top_n, index, bm25)
for doc_id, score in ranked_bm25:
    print("Doc ID:", doc_id, "Score:", score)

print("\nRRF (TF-IDF & BM25) Ranking:")
scoring_funcs = {'bm25': bm25, 'tf_idf': tf_idf}
ranked_rrf = reciprocal_rank_fusion(first_query, top_n, index, scoring_funcs)
for doc_id, score in ranked_rrf:
    print("Doc ID:", doc_id, "Score:", score)

D) **Evaluation**

In [ ]:
# --- IR System Evaluation Functions ---

def evaluation(Q, R, D, metrics, num_retrieval, scoring_function, **kwargs):
    rankings = {}
    for query in Q:
        relevant_docs = ranking(query.title, num_retrieval, D, scoring_function)
        rankings[query.query_id] = {doc_id: float(score) for doc_id, score in relevant_docs}

    evaluator = pytrec_eval.RelevanceEvaluator(R, metrics)
    per_query_results = evaluator.evaluate(rankings)

    aggregated_metrics = {}
    for metric in metrics:
        aggregated_metrics[metric] = sum(run_result[metric] for run_result in per_query_results.values()) / len(per_query_results)

    return aggregated_metrics, per_query_results

def evaluation_reciprocal_rank_fusion(queries, qrels, I, metrics, scoring_functions, num_retrieval, y=50, **kwargs):
		results = {}
		for query in queries:
				qid = query.query_id
				scores = {}
				for scoring_func_name, scoring_func in scoring_functions.items():
						ranked_results = ranking(query.title, num_retrieval, I, scoring_func, **kwargs)
						for i, (docid, _) in enumerate(ranked_results):
								if docid not in scores:
										scores[docid] = 0
								scores[docid] += 1 / (i + y)

				results[qid] = scores

		evaluator = pytrec_eval.RelevanceEvaluator(qrels, metrics)

		per_query_results = evaluator.evaluate(results)

		aggregated_metrics = {}
		for metric in metrics:
				aggregated_metrics[metric] = sum(run_result[metric] for run_result in per_query_results.values()) / len(per_query_results)

		return aggregated_metrics, per_query_results

def evaluation_unranked(queries, qrels, I):
    per_query_metrics = {}

    for query in queries:
        qid = query.query_id

        retrieved = boolean_query(qid, I, queries)
        retrieved_set = set(retrieved)
        relevant_set = {doc for doc, rel in qrels.get(qid, {}).items() if rel > 0}

        true_positives = retrieved_set.intersection(relevant_set)
        precision = len(true_positives) / len(retrieved_set) if retrieved_set else 0.0
        recall = len(true_positives) / len(relevant_set) if relevant_set else 0.0
        f_measure = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        per_query_metrics[qid] = {
            "precision": precision,
            "recall": recall,
            "f_measure": f_measure
        }

    # Aggregate metrics by averaging over all queries.
    num_queries = len(per_query_metrics)
    avg_precision = sum(m["precision"] for m in per_query_metrics.values()) / num_queries if num_queries else 0.0
    avg_recall = sum(m["recall"] for m in per_query_metrics.values()) / num_queries if num_queries else 0.0
    avg_f_measure = sum(m["f_measure"] for m in per_query_metrics.values()) / num_queries if num_queries else 0.0

    aggregated_metrics = {
        "precision": avg_precision,
        "recall": avg_recall,
        "f_measure": avg_f_measure
    }

    return aggregated_metrics, per_query_metrics

In [ ]:
# --- IR System Evaluation Demo ---
metrics = {'map', 'ndcg', 'P_10', 'ndcg_cut_10'}

print("\nBoolean Evaluation:")
elapsed_time = time()
aggregated_metrics_boolean, per_query_boolean = evaluation_unranked(queries, qrels, index)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics_boolean.items():
    print(f"{metric}: {value:.4f}")

print("\nBM25 Evaluation:")
elapsed_time = time()
aggregated_metrics, per_query_results = evaluation(queries, qrels, index, metrics, 1000, bm25)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nRRF Evaluation:")
scoring_funcs = {'bm25': bm25, 'tf_idf': tf_idf}
elapsed_time = time()
aggregated_metrics, per_query_results = evaluation_reciprocal_rank_fusion(queries, qrels, index, metrics, scoring_funcs, 1000)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics.items():
    print(f"{metric}: {value:.4f}")

<H3>Part II: questions materials (optional)</H3>

**(1)** Characterize the document collection *D* and topic collection *Q*. What is the distribution of informative terms in *D* before and after text processing?

In [ ]:
# --- Q1 | CORPUS ---

# Documents Characterization
def doc_characterization(D, I):
		doc_lengths = list(I.doc_len.values())
		avg_doc_len = I.avg_doc_len
		max_doc_len = max(doc_lengths)
		min_doc_len = min(doc_lengths)

		return avg_doc_len, max_doc_len, min_doc_len

avg_doc_len, max_doc_len, min_doc_len = doc_characterization(dataset, index)
print(f"Number of documents: {len(index.doc_len)}")
print(f"Average document length: {avg_doc_len:.2f}")
print(f"Maximum document length: {max_doc_len}")
print(f"Minimum document length: {min_doc_len}")

# Document Term Characterization
def doc_term_characterization(D, I):
		term_freq = Counter()
		for term, doc_dict in I.postings.items():
				term_freq[term] = sum(doc_dict.values())
		return term_freq

term_freq = doc_term_characterization(dataset, index)
print(f"Number of unique terms: {len(term_freq)}")
print(f"Most frequent term: {term_freq.most_common(1)[0]}")
print(f"Least frequent term: {term_freq.most_common()[-1]}")

# BARPLOT where x axis is the term name and y axis is frequency
l_term_freq = sorted(term_freq.items(), key=lambda x: x[1], reverse=True)

plt.figure(figsize=(10, 6))
plt.bar([term for term, freq in l_term_freq[:10]], [freq for term, freq in l_term_freq[:10]])
plt.xlabel("Term")
plt.ylabel("Frequency")
plt.title("Top 10 Most Frequent Terms")
plt.tight_layout()
plt.show()



################## Without Preprocessing ##################
index2, preprocessing_time, indexing_time, space_used = indexing(docs, stopwords_lang=None, preprocess_text=False)
print(f"Preprocessing time: {preprocessing_time:.4f} seconds")
print(f"Indexing time: {indexing_time:.4f} seconds")
print(f"Approximate space used: {space_used / (1024 * 1024):.2f} MB")

avg_doc_len, max_doc_len, min_doc_len = doc_characterization(dataset, index2)
print(f"Number of documents: {len(index2.doc_len)}")
print(f"Average document length: {avg_doc_len:.2f}")
print(f"Maximum document length: {max_doc_len}")
print(f"Minimum document length: {min_doc_len}")

term_freq = doc_term_characterization(dataset, index2)
print(f"Number of unique terms: {len(term_freq)}")
print(f"Most frequent term: {term_freq.most_common(1)[0]}")
print(f"Least frequent term: {term_freq.most_common()[-1]}")

# BARPLOT where x axis is the term name and y axis is frequency
l_term_freq = sorted(term_freq.items(), key=lambda x: x[1], reverse=True)

plt.figure(figsize=(10, 6))
plt.bar([term for term, freq in l_term_freq[:10]], [freq for term, freq in l_term_freq[:10]])
plt.xlabel("Term")
plt.ylabel("Frequency")
plt.title("Top 10 Most Frequent Terms")
plt.tight_layout()
plt.show()

In [ ]:
# --- Q1 | QUERIES ---

# Queries Characterization
def query_characterization(Q):
		query_lengths = [len(query.split()) for query in Q]
		avg_query_len = sum(query_lengths) / len(Q)
		max_query_len = max(query_lengths)
		min_query_len = min(query_lengths)

		return avg_query_len, max_query_len, min_query_len

queries_eval = [query.title for query in dataset.queries_iter()]

avg_query_len, max_query_len, min_query_len = query_characterization(queries_eval)
print(f"Number of queries: {len(queries_eval)}")
print(f"Average query length: {avg_query_len:.2f}")
print(f"Maximum query length: {max_query_len}")
print(f"Minimum query length: {min_query_len}")

# Query Term Characterization
def query_term_characterization(Q):
		term_freq = Counter()
		for query in Q:
				term_freq.update(query.split())
		return term_freq

term_freq = query_term_characterization(queries_eval)
print(f"Number of unique terms: {len(term_freq)}")

# BARPLOT where x axis is the term name and y axis is frequency
l_term_freq = sorted(term_freq.items(), key=lambda x: x[1], reverse=True)

plt.figure(figsize=(10, 6))
plt.bar([term for term, freq in l_term_freq[:10]], [freq for term, freq in l_term_freq[:10]])
plt.xlabel("Term")
plt.ylabel("Frequency")
plt.title("Top 10 Most Frequent Terms")
plt.tight_layout()
plt.show()

################## With Preprocessing ##################

queries_preprocessed = [preprocess(query) for query in queries_eval]

# Queries Characterization
def processed_query_characterization(Q):
		query_lengths = [len(query) for query in Q]
		avg_query_len = sum(query_lengths) / len(Q)
		max_query_len = max(query_lengths)
		min_query_len = min(query_lengths)

		return avg_query_len, max_query_len, min_query_len

avg_query_len, max_query_len, min_query_len = processed_query_characterization(queries_preprocessed)
print(f"Number of queries: {len(queries_eval)}")
print(f"Average query length: {avg_query_len:.2f}")
print(f"Maximum query length: {max_query_len}")
print(f"Minimum query length: {min_query_len}")

# Query Term Characterization
def processed_query_term_characterization(Q):
		term_freq = Counter()
		for query in Q:
				term_freq.update(query)
		return term_freq

term_freq = processed_query_term_characterization(queries_preprocessed)
print(f"Number of unique terms: {len(term_freq)}")

# BARPLOT where x axis is the term name and y axis is frequency
l_term_freq = sorted(term_freq.items(), key=lambda x: x[1], reverse=True)

plt.figure(figsize=(10, 6))
plt.bar([term for term, freq in l_term_freq[:10]], [freq for term, freq in l_term_freq[:10]])
plt.xlabel("Term")
plt.ylabel("Frequency")
plt.title("Top 10 Most Frequent Terms")
plt.tight_layout()
plt.show()

**(2)** Characterize the performance of the IR system. How much time and space are necessary for each stage - processing, indexing and retrieval - of the IR process?

In [ ]:
# --- Q2 ---

# Processing time & space
start_time = time()
space_used = 0
for doc in docs:
    docid = doc.doc_id

    # Use title + abstract for indexing.
    doc_text = f"{doc.title} {doc.abstract}"
    processed_text = preprocess(doc_text,  set(stopwords.words("english")))
    space_used += sys.getsizeof(processed_text)

print(f"Took {time() - start_time:.4f} seconds for processing stage")
print(f"Approximate space used: {space_used / (1024 * 1024):.2f} MB for processing stage")

# Indexing time & space
index3, preprocessing_time, indexing_time, space_used = indexing(docs)
print(f"Took {indexing_time:.4f} seconds for indexing stage")
print(f"Approximate space used: {space_used / (1024 * 1024):.2f} MB for indexing stage")

# Retrieval time & space
start_time = time()
space_used = 0

for query in queries:
    relevant_docs = ranking(query.title, 1000, index3, bm25)
    space_used += sys.getsizeof(relevant_docs)

print(f"Took {time() - start_time:.4f} seconds for retrieval stage (for all queries)")
print(f"Approximate space used: {space_used / (1024 * 1024):.2f} MB for retrieval stage (for all queries)")

**(3)** Given a specific *p*, is the implemented IR system better at providing recall or precision guarantees? & **(4)** Which *p* should be used by the IR system if the user has a preference towards: minimizing false positives, minimizing false negatives, or maximizing true positives?

In [ ]:
# --- Q3 & Q4 ---

# Define candidate num_retrieval values.
num_retrieval_values = [10, 100, 500, 1000, 5000]

# The set of all document IDs in the collection (universe).
universe = set(index.doc_len.keys())

# Define the relevance threshold (documents with qrel > threshold are relevant).
relevance_threshold = 0

# Set of metrics for pytrec_eval.
pytrec_metrics = {'map', 'ndcg', 'P_10', 'ndcg_cut_10', 'recall_1000'}

# List to store the evaluation results.
eval_results = []

for k in num_retrieval_values:
    manual_metrics_list = []  # To store manual metrics per query.
    rankings = {}             # To build run for pytrec_eval.

    for query in queries:
        # Retrieve ranked results for the query using the ranking function.
        ranked_results = ranking(query.title, k, index, bm25)
        retrieved_ids = {doc_id for doc_id, _ in ranked_results}

        # Build the run dictionary for pytrec_eval.
        rankings[query.query_id] = {doc_id: float(score) for doc_id, score in ranked_results}

        # Get the set of relevant document IDs (based on the qrels).
        relevant_ids = {doc_id for doc_id, rel in qrels.get(query.query_id, {}).items() if rel > relevance_threshold}

        # Compute True Positives (TP), False Positives (FP), and False Negatives (FN).
        TP = len(retrieved_ids & relevant_ids)
        FP = len(retrieved_ids - relevant_ids)
        FN = len(relevant_ids - retrieved_ids)

        # For True Negatives (TN), consider the universe of documents:
        # TN = (non-relevant docs not retrieved) = (all docs not relevant) - (retrieved docs that are not relevant)
        non_relevant = universe - relevant_ids
        TN = len(non_relevant - retrieved_ids)

        # Compute manual precision and recall.
        precision_manual = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall_manual = TP / (TP + FN) if (TP + FN) > 0 else 0.0

        manual_metrics_list.append({
            'TP': TP,
            'FP': FP,
            'FN': FN,
            'TN': TN,
            'precision': precision_manual,
            'recall': recall_manual
        })

    # Average the manual metrics over all queries.
    avg_TP = sum(m['TP'] for m in manual_metrics_list) / len(manual_metrics_list)
    avg_FP = sum(m['FP'] for m in manual_metrics_list) / len(manual_metrics_list)
    avg_FN = sum(m['FN'] for m in manual_metrics_list) / len(manual_metrics_list)
    avg_TN = sum(m['TN'] for m in manual_metrics_list) / len(manual_metrics_list)
    avg_precision_manual = sum(m['precision'] for m in manual_metrics_list) / len(manual_metrics_list)
    avg_recall_manual = sum(m['recall'] for m in manual_metrics_list) / len(manual_metrics_list)

    # Use pytrec_eval to compute the standard IR metrics.
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, pytrec_metrics)
    per_query_results = evaluator.evaluate(rankings)
    agg_pytrec = {}
    for metric in pytrec_metrics:
        agg_pytrec[metric] = sum(result[metric] for result in per_query_results.values()) / len(per_query_results)

    # Save all the results for the current num_retrieval value.
    eval_results.append({
        'num_retrieval': k,
        'manual': {
            'avg_precision': avg_precision_manual,
            'avg_recall': avg_recall_manual,
            'avg_TP': avg_TP,
            'avg_FP': avg_FP,
            'avg_FN': avg_FN,
            'avg_TN': avg_TN
        },
        'pytrec': agg_pytrec
    })

# Print a summary table of the results.
print("Evaluation Results for Different num_retrieval Values:")
print("k\tManual Precision\tManual Recall\tTP\tFP\tFN\tTN\tMAP\t\tnDCG\tP@10\tnDCG@10\tRecall@1000")
for res in eval_results:
    k = res['num_retrieval']
    man = res['manual']
    pytrec = res['pytrec']
    print(f"{k}\t{man['avg_precision']:.4f}\t\t{man['avg_recall']:.4f}\t\t{man['avg_TP']:.1f}\t{man['avg_FP']:.1f}\t{man['avg_FN']:.1f}\t{man['avg_TN']:.1f}\t"
          f"{pytrec['map']:.4f}\t{pytrec['ndcg']:.4f}\t{pytrec['P_10']:.4f}\t{pytrec['ndcg_cut_10']:.4f}\t{pytrec['recall_1000']:.4f}")

**(5)** Does performance strongly vary across topics in *Q* ? Given the available relevance judgments, can we place a general consideration on the adequacy of the ranking model / retrieval function? & **(6)** How different text processing and scoring options affect retrieval? Is reciprocal rank fusion useful to place ensemble decisions?

In [ ]:
# --- Q5 & Q6 ---
import matplotlib.pyplot as plt
import matplotlib as mpl



############ BOOLEAN ############
print("\nBOOLEAN:")

# Evaluate the IR system
elapsed_time = time()
aggregated_metrics_boolean, per_query_boolean = evaluation_unranked(queries, qrels, index)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

# Print results
print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics_boolean.items():
    print(f"{metric}: {value:.4f}")

# Update plot style parameters.
mpl.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

# Define the metrics of interest.
metric_names = ['precision', 'recall', 'f_measure']
metric_values = {metric: [] for metric in metric_names}

# Extract metric values for each query.
for qid, m_dict in per_query_boolean.items():
    for metric in metric_names:
        metric_values[metric].append(m_dict[metric])

# Order metrics as desired.
ordered_metrics = sorted(metric_values.keys())
data = [metric_values[m] for m in ordered_metrics]

# Create the boxplot.
fig, ax = plt.subplots(figsize=(3.5, 2))
bp = ax.boxplot(data, patch_artist=True)

# Style the boxes and medians.
for box in bp['boxes']:
    box.set(facecolor='lightblue', alpha=0.8)
for median in bp['medians']:
    median.set(color='navy', linewidth=1.5)

# Set axis labels.
ax.set_xticklabels(ordered_metrics)
ax.set_ylabel("Value")
fig.tight_layout()
plt.show()



############ TF-IDF ############
print("\nTF-IDF:")

metrics = {'map', 'ndcg', 'P_10', 'ndcg_cut_10'}

# Evaluate the IR system
elapsed_time = time()
aggregated_metrics, per_query_results = evaluation(queries, qrels, index, metrics, 1000, tf_idf)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

# Print results
print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics.items():
    print(f"{metric}: {value:.4f}")

mpl.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

metric_values = {metric: [] for metric in metrics}
for qid, metric_dict in per_query_results.items():
    for metric in metrics:
        metric_values[metric].append(metric_dict[metric])

ordered_metrics = sorted(metric_values.keys())
data = [metric_values[m] for m in ordered_metrics]

fig, ax = plt.subplots(figsize=(3.5, 2))
bp = ax.boxplot(data, patch_artist=True)

for box in bp['boxes']:
    box.set(facecolor='lightblue', alpha=0.8)
for median in bp['medians']:
    median.set(color='navy', linewidth=1.5)

ax.set_xticklabels(ordered_metrics)
ax.set_ylabel("Value")
fig.tight_layout()
plt.show()


############ BM25 ############
print("\nBM25:")

metrics = {'map', 'ndcg', 'P_10', 'ndcg_cut_10'}

# Evaluate the IR system
elapsed_time = time()
aggregated_metrics, per_query_results = evaluation(queries, qrels, index, metrics, 1000, bm25)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

# Print results
print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics.items():
    print(f"{metric}: {value:.4f}")

mpl.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

metric_values = {metric: [] for metric in metrics}
for qid, metric_dict in per_query_results.items():
    for metric in metrics:
        metric_values[metric].append(metric_dict[metric])

ordered_metrics = sorted(metric_values.keys())
data = [metric_values[m] for m in ordered_metrics]

fig, ax = plt.subplots(figsize=(3.5, 2))
bp = ax.boxplot(data, patch_artist=True)

for box in bp['boxes']:
    box.set(facecolor='lightblue', alpha=0.8)
for median in bp['medians']:
    median.set(color='navy', linewidth=1.5)

ax.set_xticklabels(ordered_metrics)
ax.set_ylabel("Value")
fig.tight_layout()
plt.show()


############ RRF (TF-IDF & BM25) ############
print("\nRRF (TF-IDF & BM25):")

scoring_funcs = {'bm25': bm25, 'tf_idf': tf_idf}

elapsed_time = time()
aggregated_metrics, per_query_results = evaluation_reciprocal_rank_fusion(queries, qrels, index, metrics, scoring_funcs, 1000)
elapsed_time = time() - elapsed_time
print(f"Took {elapsed_time:.4f} seconds to evaluate the IR system, {elapsed_time / len(queries):.4f} seconds per query")

# Print results
print("Aggregated Evaluation Metrics:")
for metric, value in aggregated_metrics.items():
    print(f"{metric}: {value:.4f}")

mpl.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

metric_values = {metric: [] for metric in metrics}
for qid, metric_dict in per_query_results.items():
    for metric in metrics:
        metric_values[metric].append(metric_dict[metric])

ordered_metrics = sorted(metric_values.keys())
data = [metric_values[m] for m in ordered_metrics]

fig, ax = plt.subplots(figsize=(3.5, 2))
bp = ax.boxplot(data, patch_artist=True)

for box in bp['boxes']:
    box.set(facecolor='lightblue', alpha=0.8)
for median in bp['medians']:
    median.set(color='navy', linewidth=1.5)

ax.set_xticklabels(ordered_metrics)
ax.set_ylabel("Value")
fig.tight_layout()
plt.show()

<H3>END</H3>